# Base model

In [3]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
dataset_path="dataset-update.csv"
raw_clauses_path = "output/clauses.txt"
try:
    df = pd.read_csv(dataset_path, sep=",", quotechar='"') 
except FileNotFoundError:
    raise Exception(f"Lỗi: Không tìm thấy file {dataset_path}")

In [5]:
X_text = df['text']
y_labels = df['intent']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=1000)
X_train_features = vectorizer.fit_transform(X_train_text)
X_test_features = vectorizer.transform(X_test_text)

In [6]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_features, y_train)

print("Báo cáo độ chính xác:")
test_predictions = model.predict(X_test_features)
print(classification_report(y_test, test_predictions, zero_division=0))

Báo cáo độ chính xác:
                       precision    recall  f1-score   support

           Obligation       0.62      1.00      0.76        16
          Prohibition       1.00      0.50      0.67         4
                Right       0.67      0.29      0.40         7
Termination Condition       0.00      0.00      0.00         4

             accuracy                           0.65        31
            macro avg       0.57      0.45      0.46        31
         weighted avg       0.60      0.65      0.57        31



In [7]:
if not os.path.exists(raw_clauses_path):
    raise Exception(f"Lỗi: Không tìm thấy file {raw_clauses_path}.")


with open(raw_clauses_path, 'r', encoding='utf-8') as f:
    unseen_clauses = [line.strip() for line in f.readlines() if line.strip()]

In [8]:
print(f"Đang dự đoán cho {len(unseen_clauses)} mệnh đề từ file {raw_clauses_path}...")

X_unseen_features = vectorizer.transform(unseen_clauses)
unseen_predictions = model.predict(X_unseen_features)

os.makedirs("output", exist_ok=True)
output_path = "output/intent_classification_baseline.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for clause, intent in zip(unseen_clauses, unseen_predictions):
        f.write(f"{clause}\n{intent}\n\n")
        
print(f"Đã ghi kết quả vào {output_path}")

Đang dự đoán cho 166 mệnh đề từ file output/clauses.txt...
Đã ghi kết quả vào output/intent_classification_baseline.txt


# Phobert model

In [9]:
import os
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
dataset_path = 'dataset-update.txt'
raw_clauses_path = 'output/clauses.txt'
    
try:
    df = pd.read_csv(dataset_path, sep=",", quotechar='"')
except Exception as e:
    raise Exception(f"Lỗi đọc file dataset: {e}")

In [11]:
le = LabelEncoder()
label_col = 'intent'
df['label_id'] = le.fit_transform(df[label_col])
num_labels = len(le.classes_)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label_id'].tolist(), test_size=0.2, random_state=42, stratify=df['label_id']
)

In [12]:
class ContractDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

In [13]:
model_name = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=256)

train_dataset = ContractDataset(train_encodings, train_labels)
val_dataset = ContractDataset(val_encodings, val_labels)


In [14]:
print("Đang tải PhoBERT Model...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
training_args = TrainingArguments(
    output_dir='./results_phobert',
    num_train_epochs=8,
    per_device_train_batch_size=8,    
    per_device_eval_batch_size=16,
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

Đang tải PhoBERT Model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 44543.28it/s]
RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
print("Bắt đầu huấn luyện mô hình...")
trainer.train()

print("\n" + "="*50)
print("Báo cáo độ chính xác trên tập Test:")
print("="*50)

predictions = trainer.predict(val_dataset)
y_pred_ids = np.argmax(predictions.predictions, axis=-1)

y_true_names = le.inverse_transform(val_labels)
y_pred_names = le.inverse_transform(y_pred_ids)
print(classification_report(y_true_names, y_pred_names, zero_division=0))

Bắt đầu huấn luyện mô hình...


c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.308682,1.128494
2,1.022366,0.770365
3,0.733984,0.403661
4,0.274431,0.145552
5,0.120991,0.071625
6,0.079490,0.051345
7,0.055221,0.041587
8,0.053936,0.039511


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]
c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]
c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]
c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00


Báo cáo độ chính xác trên tập Test:


                       precision    recall  f1-score   support

           Obligation       1.00      1.00      1.00        16
          Prohibition       1.00      1.00      1.00         4
                Right       1.00      1.00      1.00         7
Termination Condition       1.00      1.00      1.00         4

             accuracy                           1.00        31
            macro avg       1.00      1.00      1.00        31
         weighted avg       1.00      1.00      1.00        31



In [16]:
if not os.path.exists(raw_clauses_path):
    raise Exception(f"Lỗi: Không tìm thấy file {raw_clauses_path}")

with open(raw_clauses_path, 'r', encoding='utf-8') as f:
    unseen_clauses = [line.strip() for line in f.readlines() if line.strip()]
        
unseen_encodings = tokenizer(unseen_clauses, truncation=True, padding=True, max_length=256)
unseen_dataset = ContractDataset(unseen_encodings)

unseen_preds = trainer.predict(unseen_dataset)
unseen_pred_ids = np.argmax(unseen_preds.predictions, axis=-1)
unseen_pred_names = le.inverse_transform(unseen_pred_ids)

os.makedirs("output", exist_ok=True)
out_file = "output/phobert_intent_classification.txt"

with open(out_file, "w", encoding="utf-8") as f:
    for clause, intent in zip(unseen_clauses, unseen_pred_names):
        f.write(f"{clause}\n{intent}\n\n")
        
print(f"Đã lưu kết quả vào {out_file}")

c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Đã lưu kết quả vào output/phobert_intent_classification.txt


# Compare models

In [17]:
import os
import pandas as pd

In [18]:
def load_predictions(filepath):
    """
    Hàm đọc file kết quả theo chuẩn format:
    [Text]
    [Intent]
    <dòng trống>
    """
    if not os.path.exists(filepath):
        print(f"Không tìm thấy file: {filepath}")
        return []
        
    with open(filepath, 'r', encoding='utf-8') as f:
        blocks = f.read().strip().split('\n\n')
        
    data = []
    for block in blocks:
        lines = block.split('\n')
        if len(lines) >= 2:
            text = lines[0].strip()
            intent = lines[-1].strip() # Lấy dòng cuối cùng trong block làm intent
            data.append({"text": text, "intent": intent})
            
    return data

In [19]:
print("Tiến hành phân tích đối chiếu 2 mô hình:\n")

baseline_path = 'output/intent_classification_baseline.txt'
phobert_path = 'output/phobert_intent_classification.txt'

baseline_data = load_predictions(baseline_path)
phobert_data = load_predictions(phobert_path)

if not baseline_data or not phobert_data:
    raise Exception("File ko tìm thấy")
    
if len(baseline_data) != len(phobert_data):
    raise Exception(f"""Cảnh báo: Số lượng câu trong 2 file không khớp nhau!\n
                    Baseline: {len(baseline_data)} câu | PhoBERT: {len(phobert_data)} câu""")

Tiến hành phân tích đối chiếu 2 mô hình:



In [20]:
total_clauses = len(baseline_data)
agreements = 0
disagreements = []

# So sánh từng câu
for i in range(total_clauses):
    text = baseline_data[i]['text']
    base_intent = baseline_data[i]['intent']
    pho_intent = phobert_data[i]['intent']
    
    if base_intent == pho_intent:
        agreements += 1
    else:
        disagreements.append({
            "Clause": text,
            "Baseline": base_intent,
            "PhoBERT (Transformer)": pho_intent
        })

agreement_rate = (agreements / total_clauses) * 100
print("=" * 50)
print("Báo cáo đối chiếu mô hình:")
print("=" * 50)
print(f"Tổng số mệnh đề đã dự đoán : {total_clauses}")
print(f"Số câu 2 mô hình đồng thuận : {agreements} ({agreement_rate:.2f}%)")
print(f"Số câu 2 mô hình mâu thuẫn  : {len(disagreements)} ({100 - agreement_rate:.2f}%)")
print("=" * 50)

Báo cáo đối chiếu mô hình:
Tổng số mệnh đề đã dự đoán : 166
Số câu 2 mô hình đồng thuận : 155 (93.37%)
Số câu 2 mô hình mâu thuẫn  : 11 (6.63%)


In [21]:
if disagreements:
    print("\nMột số ví dụ mô hình bất đồng quan điểm:")
    for i, item in enumerate(disagreements[:3]):
        print(f"\n[{i+1}] {item['Clause']}")
        print(f"   Baseline đoán: {item['Baseline']}")
        print(f"   PhoBERT đoán : {item['PhoBERT (Transformer)']}")
        
if disagreements:
    df_diff = pd.DataFrame(disagreements)
    out_csv = 'output/model_disagreements.csv'
    df_diff.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f"\nĐã xuất {len(disagreements)} câu mâu thuẫn ra file: {out_csv}")


Một số ví dụ mô hình bất đồng quan điểm:

[1] Trong thời gian thử việc , Bên B được hưởng 85% mức lương chính thức .
   Baseline đoán: Obligation
   PhoBERT đoán : Right

[2] Bên A không được yêu cầu Bên B làm thêm quá 40 giờ mỗi tháng và 200 giờ mỗi năm .
   Baseline đoán: Obligation
   PhoBERT đoán : Prohibition

[3] Cấm Bên B làm việc cho đối thủ cạnh tranh trực tiếp của Bên A trong vòng 12 tháng sau khi nghỉ việc .
   Baseline đoán: Obligation
   PhoBERT đoán : Prohibition

Đã xuất 11 câu mâu thuẫn ra file: output/model_disagreements.csv
